In [2]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, f1_score, hamming_loss
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [3]:
CACHE_FILE = "ptbxl_preprocessed_cache.npz"
if not os.path.exists(CACHE_FILE):
    raise FileNotFoundError(
        f"Cache file '{CACHE_FILE}' not found. Please run pre_processing.ipynb first!"
    )

print(f"Loading cached dataset from {CACHE_FILE}...")
cached = np.load(CACHE_FILE, allow_pickle=True)

X_train = cached["X_train"]          # Raw 100 Hz 3-lead signals: (N_train, 1000, 3)
y_train = cached["y_train"]          # Binarized 5 superclasses: (N_train, 5)
X_test = cached["X_test"]            # Raw 100 Hz 3-lead signals: (N_test, 1000, 3)
y_test = cached["y_test"]            # Binarized 5 superclasses: (N_test, 5)
class_names = cached["class_names"]  # ['CD', 'HYP', 'MI', 'NORM', 'STTC']

X_feat_train = cached["X_feat_train"]  # Morphological & HRV features: (N_train, D)
X_feat_test = cached["X_feat_test"]    # Morphological & HRV features: (N_test, D)

print(f"Loaded successfully in seconds!")
print(f"Signals   -> Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features  -> Train: {X_feat_train.shape} | Test: {X_feat_test.shape}")
print(f"Targets   -> Classes ({len(class_names)}): {class_names.tolist()}")

Loading cached dataset from ptbxl_preprocessed_cache.npz...
Loaded successfully in seconds!
Signals   -> Train: (19230, 1000, 3) | Test: (2158, 1000, 3)
Features  -> Train: (19230, 36) | Test: (2158, 36)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']


In [4]:
all_predictions = {}

# --- 1. Multi-Output XGBoost ---
print("Training XGBoost Multi-Output Baseline...")
xgb_clf = MultiOutputClassifier(
    xgb.XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ),
    n_jobs=-1
)
xgb_clf.fit(X_feat_train, y_train)

# Predict probabilities for each class
xgb_probs = np.array(xgb_clf.predict_proba(X_feat_test))[:, :, 1].T
all_predictions["XGBoost (Tabular Limb)"] = xgb_probs
print(f"XGBoost Macro AUC: {roc_auc_score(y_test, xgb_probs, average='macro'):.4f}")

# --- 2. Multi-Output Random Forest ---
print("\nTraining Random Forest Multi-Output Baseline...")
rf_clf = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ),
    n_jobs=-1
)
rf_clf.fit(X_feat_train, y_train)

rf_probs = np.array(rf_clf.predict_proba(X_feat_test))[:, :, 1].T
all_predictions["Random Forest (Tabular Limb)"] = rf_probs
print(f"Random Forest Macro AUC: {roc_auc_score(y_test, rf_probs, average='macro'):.4f}")

Training XGBoost Multi-Output Baseline...
XGBoost Macro AUC: 0.7952

Training Random Forest Multi-Output Baseline...
Random Forest Macro AUC: 0.7862


In [5]:
import numpy as np
import joblib

# Save predictions for the presentation plots
np.savez_compressed("preds_classical.npz", xgb=xgb_probs, rf=rf_probs)

# Save the classical models (weights/trees)
joblib.dump(xgb_clf, "xgb_model.joblib")
joblib.dump(rf_clf, "rf_model.joblib")
print("Saved Classical predictions and models!")

Saved Classical predictions and models!
